# Pressure Pad — Live Heatmap

Visualises the Velostat pressure matrix from an Arduino GIGA R1 WiFi running `pressure_pad_data.ino`.

**Primary path — WiFi/UDP:** Arduino streams packets over the local network; no USB cable needed.  
**Fallback path — USB serial:** original wired approach; use if WiFi is unavailable.

| | Transport | Cell to run |
|---|---|---|
| **Primary** | WiFi UDP (port 4210) | WiFi Config → WiFi Heatmap |
| **Fallback** | USB Serial 115200 baud | Wired Config → Wired Heatmap |

In [1]:
# Install dependencies if needed
%pip install pyserial numpy matplotlib --quiet

# List available serial ports to help identify the Arduino
import serial.tools.list_ports
for p in serial.tools.list_ports.comports():
    print(p.device, '—', p.description)


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
/dev/cu.debug-console — n/a
/dev/cu.Bluetooth-Incoming-Port — n/a
/dev/cu.usbmodem101 — Giga


---
## WiFi / UDP (Primary)
Run **WiFi Config** then **WiFi Heatmap**. Your laptop must be on the same network as the Arduino.

In [ ]:
# ── WiFi / UDP Configuration (Primary) ────────────────────────────────────────
UDP_PORT  = 4210          # must match UDP_PORT in the .ino
WIFI_SSID = 'Tatonetti Lab'  # informational — shown in the heatmap cell
NUM_ROWS  = 5             # must match NUM_ROWS in the .ino
NUM_COLS  = 12            # must match NUM_COLS in the .ino
VMIN      = 0.0           # colormap lower bound (V)
VMAX      = 3.3           # colormap upper bound (V)
print(f'Config: {NUM_ROWS} rows × {NUM_COLS} cols | UDP {UDP_PORT} | {VMIN}–{VMAX} V')

In [ ]:
import socket
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

def _make_udp_socket():
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    if sys.platform != 'win32':
        sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEPORT, 1)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_BROADCAST, 1)
    sock.bind(('', UDP_PORT))
    sock.settimeout(2.0)
    return sock

def _read_frame_udp(sock):
    """Assemble one complete frame from UDP packets. Returns ndarray or None."""
    matrix   = np.zeros((NUM_ROWS, NUM_COLS))
    rows_rcv = set()
    frame_id = None
    deadline = time.monotonic() + 5.0

    while True:
        if time.monotonic() > deadline:
            return None
        try:
            data, _ = sock.recvfrom(512)
        except socket.timeout:
            continue

        try:
            parts = data.decode('utf-8', errors='ignore').strip().split(',')
            tag   = parts[0]

            if tag == 'FRAME':
                frame_id = int(parts[1])
                rows_rcv.clear()
                matrix[:] = 0.0
                deadline  = time.monotonic() + 5.0   # reset on fresh frame

            elif tag == 'ROW' and frame_id is not None:
                if int(parts[1]) != frame_id:
                    continue                           # stale packet
                row_idx = int(parts[2])               # 0-based
                vals    = [float(v) for v in parts[3:]]
                if len(vals) == NUM_COLS and 0 <= row_idx < NUM_ROWS:
                    matrix[row_idx] = vals
                    rows_rcv.add(row_idx)

            elif tag == 'END' and frame_id is not None:
                if int(parts[1]) == frame_id and rows_rcv:
                    return matrix                      # render on END

        except (ValueError, IndexError):
            continue                                   # malformed packet


# ── Build figure (shared by the live loop) ────────────────────────────────────
_fig_w = max(7, NUM_COLS * 1.3)
_fig, _ax = plt.subplots(figsize=(_fig_w, 5.5))
plt.close(_fig)

_im = _ax.imshow(
    np.zeros((NUM_ROWS, NUM_COLS)),
    vmin=VMIN, vmax=VMAX,
    cmap='plasma', aspect='auto', origin='lower',
)
_cbar = _fig.colorbar(_im, ax=_ax, fraction=0.03, pad=0.03)
_cbar.set_label('Voltage (V)', fontsize=11)

_ax.set_xticks(range(NUM_COLS))
_ax.set_xticklabels([f'Col {i}' for i in range(1, NUM_COLS + 1)], fontsize=9)
_ax.set_yticks(range(NUM_ROWS))
_ax.set_yticklabels([f'Row {i}' for i in range(1, NUM_ROWS + 1)], fontsize=10)
_ax.set_title('Pressure Pad — Live Heatmap (WiFi)', fontsize=13, pad=10)

_cell_font  = max(6, 9 - (NUM_COLS - 4) // 2)
_cell_texts = [
    [_ax.text(j, i, '', ha='center', va='center',
              fontsize=_cell_font, fontweight='bold')
     for j in range(NUM_COLS)]
    for i in range(NUM_ROWS)
]
_fig.tight_layout()

# ── Stream ────────────────────────────────────────────────────────────────────
print(f'Listening on UDP port {UDP_PORT}  [{NUM_ROWS} rows × {NUM_COLS} cols]')
print(f'Laptop must be on the same network as the Arduino (SSID: "{WIFI_SSID}").')
print('Interrupt the kernel (stop button) to stop.\n')

udp_sock    = _make_udp_socket()
frame_count = 0
try:
    while True:
        frame = _read_frame_udp(udp_sock)
        if frame is None:
            print('No frame received within 5 s — check WiFi / UDP port.')
            break

        frame_count += 1
        _im.set_data(frame)

        for i in range(NUM_ROWS):
            for j in range(NUM_COLS):
                v   = frame[i, j]
                txt = _cell_texts[i][j]
                txt.set_text(f'{v:.2f}V')
                txt.set_color('k' if v > VMAX * 0.55 else 'w')

        _ax.set_title(
            f'Pressure Pad — Live Heatmap (WiFi)   [{time.strftime("%H:%M:%S")}]'
            f'  frame #{frame_count}',
            fontsize=12, pad=10,
        )
        clear_output(wait=True)
        display(_fig)

except KeyboardInterrupt:
    print(f'\nStopped after {frame_count} frames.')
finally:
    udp_sock.close()
    plt.close('all')
    print('Socket closed.')

# Emergency cleanup — run this if any loop above is stuck

# Close WiFi socket
try:
    udp_sock.close()
    print('UDP socket closed.')
except Exception as e:
    print(f'udp_sock: {e}')

# Close serial port (wired fallback)
try:
    ser.close()
    print('Serial port closed.')
except Exception as e:
    print(f'ser: {e}')

plt.close('all')

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
import serial.tools.list_ports as _lp

BAUD = 115200
VMIN = 0.0    # colormap lower bound (V)
VMAX = 3.3    # colormap upper bound (V)

# Auto-detect the Arduino: picks the first cu.usbmodem* port on macOS.
# Override PORT manually below if you have multiple devices connected.
PORT = None
_candidates = [p.device for p in _lp.comports() if 'usbmodem' in p.device]
if _candidates:
    PORT = _candidates[0]
    print(f'Auto-detected port: {PORT}')
    if len(_candidates) > 1:
        print(f'Other candidates: {_candidates[1:]} — set PORT manually if wrong')
else:
    _all = [f'  {p.device}  ({p.description})' for p in _lp.comports()]
    msg  = 'No usbmodem port found. Available ports:\n' + ('\n'.join(_all) or '  (none)')
    raise RuntimeError(msg + '\n\nPlug in the Arduino, or set PORT manually above.')

Auto-detected port: /dev/cu.usbmodem101


In [ ]:
# ── Serial Diagnostics — run this to see raw lines before the heatmap ─────────
# Prints 30 raw lines so you can confirm the format matches what the parser expects.
import serial, time

_diag = serial.Serial(PORT, BAUD, timeout=2)
time.sleep(2)
_diag.reset_input_buffer()
print(f'--- first 30 lines from {PORT} ---')
for _ in range(30):
    line = _diag.readline().decode('utf-8', errors='replace').rstrip('\r\n')
    print(repr(line))
_diag.close()
print('--- done ---')

In [ ]:
import re
import time
import serial
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Self-contained defaults — override by running WiFi Config first
_VMIN = globals().get('VMIN', 0.0)
_VMAX = globals().get('VMAX', 3.3)
NUM_ROWS = 5
NUM_COLS = 12   # match NUM_COLS in the .ino

COL_LABELS = [f'Col {i}' for i in range(1, NUM_COLS + 1)]
ROW_LABELS = [f'Row {i}' for i in range(1, NUM_ROWS + 1)]

_ROW_PREFIX = re.compile(r'Row (\d+):')
_VOLT_RE    = re.compile(r'(-?[\d.]+)V')

def _read_frame_serial(ser):
    matrix     = np.zeros((NUM_ROWS, NUM_COLS))
    rows_found = set()
    deadline   = time.monotonic() + 4.0
    while len(rows_found) < NUM_ROWS:
        if time.monotonic() > deadline:
            return None
        try:
            raw = ser.readline()
        except serial.SerialException:
            return None
        line = raw.decode('utf-8', errors='ignore').strip()
        m = _ROW_PREFIX.match(line)
        if not m:
            continue
        row_num = int(m.group(1))
        vals    = [float(v) for v in _VOLT_RE.findall(line)]
        if len(vals) != NUM_COLS:
            continue
        matrix[row_num - 1] = vals
        rows_found.add(row_num)
    return matrix


fig_w = max(7, NUM_COLS * 1.3)
fig, ax = plt.subplots(figsize=(fig_w, 5.5))
plt.close(fig)

im = ax.imshow(
    np.zeros((NUM_ROWS, NUM_COLS)),
    vmin=_VMIN, vmax=_VMAX,
    cmap='plasma', aspect='auto', origin='lower',
)
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
cbar.set_label('Voltage (V)', fontsize=11)

ax.set_xticks(range(NUM_COLS))
ax.set_xticklabels(COL_LABELS, fontsize=9)
ax.set_yticks(range(NUM_ROWS))
ax.set_yticklabels(ROW_LABELS, fontsize=10)
ax.set_title('Pressure Pad — Live Heatmap (Wired)', fontsize=13, pad=10)

cell_font  = max(6, 9 - (NUM_COLS - 4) // 2)
cell_texts = [
    [ax.text(j, i, '', ha='center', va='center', fontsize=cell_font, fontweight='bold')
     for j in range(NUM_COLS)]
    for i in range(NUM_ROWS)
]
fig.tight_layout()

print(f'Opening {PORT} at {BAUD} baud  [{NUM_ROWS} rows × {NUM_COLS} cols] ...')
ser = serial.Serial(PORT, BAUD, timeout=2)
time.sleep(2)
ser.reset_input_buffer()
print('Connected. Interrupt the kernel (stop button) to stop.\n')

frame_count = 0
try:
    while True:
        frame = _read_frame_serial(ser)
        if frame is None:
            print('Timeout waiting for frame — check cable / port.')
            break

        frame_count += 1
        im.set_data(frame)

        for i in range(NUM_ROWS):
            for j in range(NUM_COLS):
                v   = frame[i, j]
                txt = cell_texts[i][j]
                txt.set_text(f'{v:.2f}V')
                txt.set_color('k' if v > _VMAX * 0.55 else 'w')

        ax.set_title(
            f'Pressure Pad — Live Heatmap (Wired)   [{time.strftime("%H:%M:%S")}]'
            f'  frame #{frame_count}',
            fontsize=12, pad=10,
        )
        clear_output(wait=True)
        display(fig)

except KeyboardInterrupt:
    print(f'\nStopped after {frame_count} frames.')
finally:
    ser.close()
    plt.close('all')
    print('Serial port closed.')

In [ ]:
# Emergency cleanup — run this if the loop cell is stuck
try:
    ser.close()
    print('Serial port closed.')
except Exception as e:
    print(e)
plt.close('all')